In [12]:
import pandas as pd

# =========================
# LOAD DATA
# =========================
df = pd.read_csv("students.csv")

# =========================
# DROP UNUSED
# =========================
df = df.drop(columns=["age"])

# =========================
# CATEGORICAL MAPPING
# =========================
study_method_map = {
    "coaching": 1.00,
    "mixed": 0.5,
    "group study": 0.2,
    "online videos": 0.1,
    "self-study": 0.00
}

sleep_quality_map = {
    "poor": 0.0,
    "average": 0.5,
    "good": 1.0
}

facility_map = {
    "low": 0.0,
    "medium": 0.5,
    "high": 1.0
}

internet_map = {
    "no": 0.0,
    "yes": 1.0
}

difficulty_map = {
    "hard": 0.0,
    "moderate": 0.5,
    "easy": 1.0
}

df["study_method"] = df["study_method"].map(study_method_map)
df["sleep_quality"] = df["sleep_quality"].map(sleep_quality_map)
df["facility_rating"] = df["facility_rating"].map(facility_map)
df["internet_access"] = df["internet_access"].map(internet_map)
df["exam_difficulty"] = df["exam_difficulty"].map(difficulty_map)

# =========================
# RANGE-BASED RULES
# =========================

# Study Hours
def map_study_hours(x):
    if x < 2:
        return 0.2
    elif x < 4:
        return 0.5
    elif x < 6:
        return 0.8
    else:
        return 1.0

# Attendance
def map_attendance(x):
    if x < 60:
        return 0.2
    elif x < 75:
        return 0.5
    elif x < 90:
        return 0.8
    else:
        return 1.0

# Sleep Hours (with optimal zone)
def map_sleep_hours(x):
    if x < 5:
        return 0.2
    elif x < 6.5:
        return 0.5
    elif x <= 8:
        return 1.0
    else:
        return 0.7  # slight penalty for oversleep

df["study_hours"] = df["study_hours"].apply(map_study_hours)
df["class_attendance"] = df["class_attendance"].apply(map_attendance)
df["sleep_hours"] = df["sleep_hours"].apply(map_sleep_hours)

# =========================
# WEIGHTS
# =========================
weights = {
    "internet_access": 0.0054,
    "sleep_quality": 0.0420,
    "study_method": 0.0451,
    "facility_rating": 0.0300,
    "exam_difficulty": 0.0122,
    "study_hours": 0.5755,
    "class_attendance": 0.1549,
    "sleep_hours": 0.0679
}

# =========================
# SAW CALCULATION
# =========================
df["saw_score"] = 0

for col, w in weights.items():
    df["saw_score"] += df[col] * w

# Optional: scale to 0–100
df["saw_score_100"] = df["saw_score"] * 100

# =========================
# OUTPUT
# =========================
print(df[["student_id", "saw_score", "saw_score_100"]].head())

# Save result
df.to_csv("student_with_saw.csv", index=False)

   student_id  saw_score  saw_score_100
0           1    0.56105         56.105
1           2    0.43079         43.079
2           3    0.83355         83.355
3           4    0.21704         21.704
4           5    0.29668         29.668


In [13]:
# =========================
# PASS / FAIL (threshold = 60)
# =========================
PASSING_GRADE = 60

# Actual result
df["actual_pass"] = df["exam_score"] >= PASSING_GRADE

# SAW prediction
df["saw_pass"] = df["saw_score_100"] >= PASSING_GRADE

# =========================
# COMPARISON
# =========================
df["match"] = df["actual_pass"] == df["saw_pass"]

# Show sample
print(df[[
    "student_id",
    "exam_score",
    "saw_score_100",
    "actual_pass",
    "saw_pass",
    "match"
]].head(10))

# =========================
# CONFUSION MATRIX
# =========================
TP = ((df["actual_pass"] == True) & (df["saw_pass"] == True)).sum()
TN = ((df["actual_pass"] == False) & (df["saw_pass"] == False)).sum()
FP = ((df["actual_pass"] == False) & (df["saw_pass"] == True)).sum()
FN = ((df["actual_pass"] == True) & (df["saw_pass"] == False)).sum()

print("\nConfusion Matrix:")
print(f"TP (Correct Pass)     : {TP}")
print(f"TN (Correct Fail)     : {TN}")
print(f"FP (Wrong Pass)       : {FP}")
print(f"FN (Wrong Fail)       : {FN}")

# =========================
# METRICS
# =========================
accuracy = (TP + TN) / len(df)
precision = TP / (TP + FP) if (TP + FP) != 0 else 0
recall = TP / (TP + FN) if (TP + FN) != 0 else 0

print("\nMetrics:")
print(f"Accuracy  : {accuracy:.4f}")
print(f"Precision : {precision:.4f}")
print(f"Recall    : {recall:.4f}")

   student_id  exam_score  saw_score_100  actual_pass  saw_pass  match
0           1        58.9         56.105        False     False   True
1           2        54.8         43.079        False     False   True
2           3        90.3         83.355         True      True   True
3           4        29.7         21.704        False     False   True
4           5        43.7         29.668        False     False   True
5           6        58.2         45.483        False     False   True
6           7        53.7         32.136        False     False   True
7           8        47.3         55.041        False     False   True
8           9        44.9         52.815        False     False   True
9          10        77.7         72.553         True      True   True

Confusion Matrix:
TP (Correct Pass)     : 8088
TN (Correct Fail)     : 7842
FP (Wrong Pass)       : 1111
FN (Wrong Fail)       : 2959

Metrics:
Accuracy  : 0.7965
Precision : 0.8792
Recall    : 0.7321
